In [1]:
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
import time
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, GradientBoostingClassifier, AdaBoostClassifier, StackingClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score, 
                             f1_score, precision_score, recall_score, precision_recall_fscore_support,
                             roc_auc_score, roc_curve)
from sklearn.preprocessing import label_binarize
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import hstack, csr_matrix
import joblib
import warnings
import os
warnings.filterwarnings('ignore')

# Check scikit-learn version
import sklearn
print(f"Scikit-learn version: {sklearn.__version__}")
print(f"Number of CPU cores: {os.cpu_count()}")

# Set style for better visualizations
plt.style.use('default')
sns.set_palette("husl")

# Create output directory
output_dir = 'sentiment_analysis_results'
os.makedirs(output_dir, exist_ok=True)

# Download necessary NLTK data
print("\nDownloading NLTK data...")
nltk.download('vader_lexicon', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)

# ==================== CUSTOM TRANSFORMER FOR EXTRA FEATURES ====================
class TextFeatureExtractor(BaseEstimator, TransformerMixin):
    """Extract additional features from text: length, VADER scores, etc."""
    
    def __init__(self):
        self.sia = SentimentIntensityAnalyzer()
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        features = []
        for text in X:
            if not isinstance(text, str) or len(text.strip()) == 0:
                features.append([0, 0, 0, 0, 0, 0, 0])
                continue
            
            # Text length (word count)
            text_len = len(text.split())
            
            # VADER scores
            vader_scores = self.sia.polarity_scores(text)
            
            # Basic text statistics
            char_count = len(text)
            word_count = len(text.split())
            avg_word_len = char_count / word_count if word_count > 0 else 0
            
            features.append([
                text_len,
                vader_scores['neg'],
                vader_scores['neu'],
                vader_scores['pos'],
                vader_scores['compound'],
                char_count,
                avg_word_len
            ])
        
        return np.array(features)

# ==================== TEXT PREPROCESSING FUNCTION ====================
def preprocess_text(text):
    """Clean text: lowercase, remove noise, lemmatize"""
    if not isinstance(text, str):
        return ""
    
    # Lowercase
    text = text.lower()
    
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tokenize and lemmatize
    lemmatizer = WordNetLemmatizer()
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stopwords.words('english')]
    
    return ' '.join(words)

# ==================== MANUAL RESAMPLING FUNCTION ====================
def manual_resample(df, label_column, target_count=None):
    """
    Manually resample data to balance classes
    """
    from collections import Counter
    
    # Get class distribution
    class_counts = df[label_column].value_counts()
    print(f"Original class distribution: {dict(class_counts)}")
    
    # Determine target count (use max class count)
    if target_count is None:
        target_count = int(class_counts.max())
    
    print(f"Target count per class: {target_count}")
    
    # Get indices for each class
    balanced_indices = []
    classes = class_counts.index.tolist()
    
    for class_name in classes:
        class_indices = df[df[label_column] == class_name].index.tolist()
        
        if len(class_indices) >= target_count:
            # If class has enough samples, randomly sample down
            sampled = np.random.choice(class_indices, size=target_count, replace=False)
        else:
            # If class has fewer samples, sample with replacement
            sampled = np.random.choice(class_indices, size=target_count, replace=True)
        
        balanced_indices.extend(sampled)
    
    # Create balanced dataframe
    df_balanced = df.loc[balanced_indices].copy()
    
    print(f"Balanced class distribution: {dict(df_balanced[label_column].value_counts())}")
    
    return df_balanced

# ==================== FUNCTION TO CALCULATE ROC AUC ====================
def calculate_roc_auc(model, X_test, y_test, classes, model_name):
    """
    Calculate ROC AUC for multi-class classification
    """
    try:
        # Check if model has predict_proba method
        if hasattr(model, "predict_proba"):
            y_score = model.predict_proba(X_test)
        elif hasattr(model, "decision_function"):
            y_score = model.decision_function(X_test)
            # Convert to probability-like scores
            y_score = 1 / (1 + np.exp(-y_score))
        else:
            return None
        
        # Binarize labels for multi-class ROC
        y_test_bin = label_binarize(y_test, classes=classes)
        
        # Calculate ROC AUC for each class (One-vs-Rest)
        roc_auc = {}
        for i, class_name in enumerate(classes):
            try:
                if len(np.unique(y_test_bin[:, i])) > 1:  # Check if class exists in test set
                    roc_auc[class_name] = roc_auc_score(y_test_bin[:, i], y_score[:, i])
                else:
                    roc_auc[class_name] = None
            except:
                roc_auc[class_name] = None
        
        # Calculate macro average
        valid_auc = [v for v in roc_auc.values() if v is not None]
        if valid_auc:
            roc_auc['macro_avg'] = np.mean(valid_auc)
        else:
            roc_auc['macro_avg'] = None
            
        return roc_auc
    except Exception as e:
        print(f"  Could not calculate ROC AUC for {model_name}: {e}")
        return None

# ==================== FIXED FUNCTION FOR NEW REVIEW PREDICTION ====================
def predict_new_review(review_text, model, tfidf_vectorizer, feature_extractor, classes, nb_tfidf=None):
    """
    Predict sentiment for a new review - Fixed to handle different model types
    """
    # Preprocess the review
    processed = preprocess_text(review_text)
    
    if not processed or len(processed.strip()) == 0:
        return {"error": "Review text is empty after preprocessing"}
    
    # Check if this is a Naive Bayes model (needs separate features)
    is_naive_bayes = 'MultinomialNB' in str(model.__class__)
    
    if is_naive_bayes and nb_tfidf is not None:
        # For Naive Bayes, use the separate TF-IDF vectorizer
        review_features = nb_tfidf.transform([processed])
    else:
        # For other models, use combined features
        review_tfidf = tfidf_vectorizer.transform([processed])
        review_extra = feature_extractor.transform([processed])
        review_features = hstack([review_tfidf, csr_matrix(review_extra)])
    
    # Make prediction
    prediction = model.predict(review_features)[0]
    
    # Get prediction probabilities if available
    probabilities = None
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(review_features)[0]
        probabilities = {classes[i]: float(probs[i]) for i in range(len(classes))}
    
    # Get VADER score for reference
    sia = SentimentIntensityAnalyzer()
    vader_score = sia.polarity_scores(review_text)['compound']
    
    result = {
        'review': review_text[:100] + '...' if len(review_text) > 100 else review_text,
        'predicted_sentiment': prediction,
        'vader_score': vader_score,
        'probabilities': probabilities,
        'model_type': 'Naive Bayes' if is_naive_bayes else 'Other'
    }
    
    return result

# ==================== FUNCTION FOR BATCH PREDICTION ====================
def predict_batch_reviews(reviews_df, text_column, model, tfidf_vectorizer, feature_extractor, classes, nb_tfidf=None):
    """
    Predict sentiment for a batch of reviews
    """
    results = []
    
    for idx, row in reviews_df.iterrows():
        review_text = row[text_column]
        result = predict_new_review(review_text, model, tfidf_vectorizer, feature_extractor, classes, nb_tfidf)
        result['index'] = idx
        results.append(result)
    
    return pd.DataFrame(results)

# ==================== 1. LOAD AND EXPLORE DATA ====================
print("\n" + "=" * 70)
print("STEP 1: LOADING AND EXPLORING DATA")
print("=" * 70)

df = pd.read_csv("capstone_airline_reviews_with_violent_2.csv")

print(f"Original shape: {df.shape}")
print(f"\nColumns in dataset:")
for i, col in enumerate(df.columns):
    print(f"  {i+1}. '{col}'")

# Check for required columns
if 'review_text' not in df.columns:
    # Try to find alternative text columns
    text_columns = [col for col in df.columns if 'review' in col.lower() or 'text' in col.lower()]
    if text_columns:
        print(f"\n'review_text' not found. Using '{text_columns[0]}' as text column")
        df.rename(columns={text_columns[0]: 'review_text'}, inplace=True)
    else:
        raise ValueError("No suitable text column found. Please check your data.")

# ==================== 2. IMPLEMENT ALGORITHM 1: VADER LABELING WITH VIOLENCE THRESHOLD ====================
print("\n" + "=" * 70)
print("STEP 2: IMPLEMENTING ALGORITHM 1 - VADER SENTIMENT WITH VIOLENCE THRESHOLD")
print("=" * 70)
print("Classes: ['positive', 'neutral', 'negative', 'violent']")
print("Violent = Very Negative (VADER compound score < -0.5)")

# Initialize VADER
sia = SentimentIntensityAnalyzer()

# Analyze VADER score distribution to set appropriate threshold
print("\nAnalyzing VADER score distribution...")
vader_scores = df['review_text'].dropna().apply(lambda x: sia.polarity_scores(str(x))['compound'])
print(f"VADER score statistics:")
print(f"  Mean: {vader_scores.mean():.3f}")
print(f"  Std: {vader_scores.std():.3f}")
print(f"  Min: {vader_scores.min():.3f}")
print(f"  Max: {vader_scores.max():.3f}")
print(f"  25th percentile: {vader_scores.quantile(0.25):.3f}")
print(f"  75th percentile: {vader_scores.quantile(0.75):.3f}")

# Define thresholds based on distribution
POSITIVE_THRESHOLD = 0.05
NEGATIVE_THRESHOLD = -0.05
VIOLENT_THRESHOLD = -0.5

def apply_vader_labeling(row):
    """
    Apply Algorithm 1 labeling with violence threshold
    """
    text = row['review_text'] if pd.notna(row['review_text']) else ""
    
    if not text or len(text.strip()) < 10:
        return None
    
    # Get VADER compound score
    vader_score = sia.polarity_scores(text)['compound']
    
    # Apply Algorithm 1 rules with violence threshold
    if vader_score > POSITIVE_THRESHOLD:
        return "positive"
    elif vader_score < NEGATIVE_THRESHOLD:
        # Check if it's violent (very negative)
        if vader_score < VIOLENT_THRESHOLD:
            return "violent"
        else:
            return "negative"
    else:
        return "neutral"

# Apply the algorithm to create labels
df['label'] = df.apply(apply_vader_labeling, axis=1)

# Remove rows where labeling failed
df = df.dropna(subset=['label'])

print("\nLabel distribution from Algorithm 1:")
label_counts = df['label'].value_counts()
print(label_counts)
print("\nLabel distribution (%):")
print(df['label'].value_counts(normalize=True) * 100)

# ==================== 3. VISUALIZATION 1: CLASS DISTRIBUTION BEFORE BALANCING ====================
print("\n" + "=" * 70)
print("STEP 3: CLASS DISTRIBUTION VISUALIZATION (BEFORE BALANCING)")
print("=" * 70)

# Figure 1: Class Distribution Before Balancing
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Colors for classes
colors = ['#2ecc71', '#f39c12', '#e74c3c', '#8e44ad']

# Pie chart
axes[0].pie(label_counts.values, labels=label_counts.index, autopct='%1.1f%%', 
            colors=colors, startangle=90, explode=(0.05, 0.05, 0.05, 0.05))
axes[0].set_title('Class Distribution Before Balancing (Pie Chart)', fontsize=14, fontweight='bold')

# Bar chart
bars = axes[1].bar(label_counts.index, label_counts.values, color=colors, edgecolor='black', linewidth=1.5)
axes[1].set_title('Class Distribution Before Balancing (Bar Chart)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Sentiment Class', fontsize=12)
axes[1].set_ylabel('Number of Reviews', fontsize=12)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height + 5,
                 f'{int(height)}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'figure1_class_distribution_before.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Figure 1 saved as '{output_dir}/figure1_class_distribution_before.png'")

# ==================== 4. VERIFY CLASSES ====================
print("\n" + "=" * 70)
print("STEP 4: VERIFYING CLASSES")
print("=" * 70)

expected_classes = ['positive', 'neutral', 'negative', 'violent']
actual_classes = sorted(df['label'].unique())
print(f"Expected classes: {expected_classes}")
print(f"Actual classes: {actual_classes}")

if set(actual_classes) == set(expected_classes):
    print("✅ Class verification passed: Exactly 4 classes")
else:
    print(f"⚠️  Class mismatch!")
    print(f"Missing: {set(expected_classes) - set(actual_classes)}")
    print(f"Extra: {set(actual_classes) - set(expected_classes)}")

# ==================== 5. PREPARE DATA FOR BALANCING ====================
print("\n" + "=" * 70)
print("STEP 5: PREPARING DATA FOR BALANCING")
print("=" * 70)

TEXT_COLUMN = "review_text"
LABEL_COLUMN = "label"

# Select only needed columns and drop NA
df_clean = df[[TEXT_COLUMN, LABEL_COLUMN]].dropna()
print(f"After dropping NA: {df_clean.shape}")

# Preprocess text
print("\nPreprocessing text...")
df_clean['processed_text'] = df_clean[TEXT_COLUMN].apply(preprocess_text)

# Remove empty or very short reviews
df_clean = df_clean[df_clean['processed_text'].str.len() > 10]
print(f"After removing very short reviews: {df_clean.shape}")

# Remove duplicates
df_clean = df_clean.drop_duplicates(subset=['processed_text'])
print(f"After removing duplicates: {df_clean.shape}")

# Final class distribution before balancing
print(f"\nClass distribution before balancing:")
final_dist_before = df_clean[LABEL_COLUMN].value_counts()
print(final_dist_before)

# Calculate imbalance ratio
min_class = final_dist_before.min()
max_class = final_dist_before.max()
imbalance_ratio = max_class / min_class
print(f"\nImbalance ratio (max/min): {imbalance_ratio:.2f}")

# ==================== 6. BALANCE THE DATA ====================
print("\n" + "=" * 70)
print("STEP 6: BALANCING THE DATA")
print("=" * 70)

# Use manual resampling (faster and more reliable)
print("\nUsing manual resampling for class balancing...")
df_balanced = manual_resample(df_clean, LABEL_COLUMN)

print(f"\nBalanced dataset shape: {df_balanced.shape}")
print(f"Balanced class distribution:")
balanced_counts = df_balanced[LABEL_COLUMN].value_counts()
print(balanced_counts)

# Save balanced data
df_balanced.to_csv('newbalanceddata.csv', index=False)
print("\n✓ Balanced data saved as 'newbalanceddata.csv'")

# ==================== 7. VISUALIZATION 2: CLASS DISTRIBUTION AFTER BALANCING ====================
print("\n" + "=" * 70)
print("STEP 7: CLASS DISTRIBUTION VISUALIZATION (AFTER BALANCING)")
print("=" * 70)

# Figure 2: Class Distribution After Balancing
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart
axes[0].pie(balanced_counts.values, labels=balanced_counts.index, autopct='%1.1f%%', 
            colors=colors, startangle=90, explode=(0.05, 0.05, 0.05, 0.05))
axes[0].set_title('Class Distribution After Balancing (Pie Chart)', fontsize=14, fontweight='bold')

# Bar chart
bars = axes[1].bar(balanced_counts.index, balanced_counts.values, color=colors, edgecolor='black', linewidth=1.5)
axes[1].set_title('Class Distribution After Balancing (Bar Chart)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Sentiment Class', fontsize=12)
axes[1].set_ylabel('Number of Reviews', fontsize=12)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height + 5,
                 f'{int(height)}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'figure2_class_distribution_after.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Figure 2 saved as '{output_dir}/figure2_class_distribution_after.png'")

# ==================== 8. TRAIN-TEST SPLIT (USING BALANCED DATA) ====================
print("\n" + "=" * 70)
print("STEP 8: TRAIN-TEST SPLIT (USING BALANCED DATA)")
print("=" * 70)

X = df_balanced['processed_text']
y = df_balanced[LABEL_COLUMN].astype(str)

print(f"Unique classes: {sorted(y.unique())}")
print(f"Number of classes: {len(y.unique())}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"\nTraining set class distribution:")
print(y_train.value_counts())
print(f"\nTest set class distribution:")
print(y_test.value_counts())

# ==================== 9. FEATURE EXTRACTION ====================
print("\n" + "=" * 70)
print("STEP 9: FEATURE EXTRACTION")
print("=" * 70)
print("Extracting: TF-IDF + basic extra features (length, VADER scores)")

# Create TF-IDF features
tfidf_vectorizer = TfidfVectorizer(
    max_features=2000,
    ngram_range=(1, 2),
    stop_words='english',
    min_df=3,
    max_df=0.9,
    sublinear_tf=True
)

# Create extra features
feature_extractor = TextFeatureExtractor()

# Transform training data
print("Transforming training data...")
start_time = time.time()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_train_extra = feature_extractor.fit_transform(X_train)
X_train_combined = hstack([X_train_tfidf, csr_matrix(X_train_extra)])
print(f"Training features shape: {X_train_combined.shape}")
print(f"Time taken: {time.time() - start_time:.2f} seconds")

# Transform test data
print("\nTransforming test data...")
start_time = time.time()
X_test_tfidf = tfidf_vectorizer.transform(X_test)
X_test_extra = feature_extractor.transform(X_test)
X_test_combined = hstack([X_test_tfidf, csr_matrix(X_test_extra)])
print(f"Test features shape: {X_test_combined.shape}")
print(f"Time taken: {time.time() - start_time:.2f} seconds")

# ==================== 10. STEP 1: MODEL TRAINING AND ENSEMBLE LEARNING ====================
print("\n" + "=" * 70)
print("STEP 10: MODEL TRAINING AND ENSEMBLE LEARNING")
print("=" * 70)
print("Training individual classifiers: RF, DT, SVM, NB")
print("Training ensemble models: Voting, Stacking, Boosting")

# ==================== 10.1 INDIVIDUAL CLASSIFIERS ====================
print("\n" + "-" * 50)
print("10.1 Individual Classifiers")
print("-" * 50)

# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    max_depth=20
)

# Decision Tree
dt_model = DecisionTreeClassifier(
    class_weight='balanced',
    random_state=42,
    max_depth=15,
    min_samples_split=5
)

# SVM (LinearSVC)
svm_model = LinearSVC(
    C=1.0,
    class_weight='balanced',
    random_state=42,
    max_iter=1000,
    dual=True,
    tol=1e-4
)

# Naive Bayes (needs separate features)
nb_tfidf = TfidfVectorizer(max_features=2000, ngram_range=(1, 2), stop_words='english')
X_train_nb = nb_tfidf.fit_transform(X_train)
X_test_nb = nb_tfidf.transform(X_test)
nb_model = MultinomialNB()

# Store individual models
individual_models = {
    'Random Forest': rf_model,
    'Decision Tree': dt_model,
    'SVM': svm_model,
    'Naive Bayes': nb_model
}

# ==================== 10.2 ENSEMBLE MODELS ====================
print("\n" + "-" * 50)
print("10.2 Ensemble Models")
print("-" * 50)

# Voting Classifier (Hard Voting)
print("\nCreating Voting Classifier (RF + DT + SVM)...")
voting_model = VotingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)),
        ('dt', DecisionTreeClassifier(class_weight='balanced', random_state=42, max_depth=15)),
        ('svm', LinearSVC(C=1.0, class_weight='balanced', random_state=42, max_iter=1000, dual=True))
    ],
    voting='hard',
    n_jobs=-1
)

# Stacking Classifier (RF, DT, SVM as base models, Logistic Regression as meta-classifier)
print("\nCreating Stacking Classifier (Base: RF, DT, SVM | Meta: Logistic Regression)...")
stacking_model = StackingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)),
        ('dt', DecisionTreeClassifier(class_weight='balanced', random_state=42, max_depth=15)),
        ('svm', LinearSVC(C=1.0, class_weight='balanced', random_state=42, max_iter=1000, dual=True))
    ],
    final_estimator=LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    cv=5,
    n_jobs=-1
)

# Boosting Models
print("\nCreating Boosting Models...")

# For scikit-learn compatibility, use different approaches based on version
if sklearn.__version__ >= '1.2':
    # For newer versions
    adaboost_dt_model = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=3, class_weight='balanced'),
        n_estimators=100,
        learning_rate=0.1,
        random_state=42,
        algorithm='SAMME'
    )
else:
    # For older versions (using base_estimator instead of estimator)
    adaboost_dt_model = AdaBoostClassifier(
        base_estimator=DecisionTreeClassifier(max_depth=3, class_weight='balanced'),
        n_estimators=100,
        learning_rate=0.1,
        random_state=42,
        algorithm='SAMME'
    )

# Gradient Boosting (works in all versions)
gradient_boost_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

# Store ensemble models
ensemble_models = {
    'Voting (Hard)': voting_model,
    'Stacking': stacking_model,
    'AdaBoost (DT)': adaboost_dt_model,
    'Gradient Boosting': gradient_boost_model
}

# Combine all models
all_models = {**individual_models, **ensemble_models}

# ==================== 11. STEP 2: PERFORMANCE EVALUATION ====================
print("\n" + "=" * 70)
print("STEP 11: PERFORMANCE EVALUATION")
print("=" * 70)
print("Metrics: Accuracy, Precision, Recall, F1-Score, ROC AUC")

# Store comprehensive results
results = {}
predictions = {}
detailed_metrics = {}
roc_auc_results = {}

# Train and evaluate each model
for name, model in all_models.items():
    print(f"\n{'='*50}")
    print(f"▶ {name}")
    print(f"{'='*50}")
    
    try:
        start_time = time.time()
        
        # Train
        if name == 'Naive Bayes':
            # Train Naive Bayes on separate features
            model.fit(X_train_nb, y_train)
            training_time = time.time() - start_time
            # Predict
            y_pred = model.predict(X_test_nb)
            # Get probabilities for ROC AUC
            if hasattr(model, "predict_proba"):
                y_score = model.predict_proba(X_test_nb)
            else:
                y_score = None
        else:
            model.fit(X_train_combined, y_train)
            training_time = time.time() - start_time
            # Predict
            y_pred = model.predict(X_test_combined)
            # Get probabilities for ROC AUC
            if hasattr(model, "predict_proba"):
                y_score = model.predict_proba(X_test_combined)
            elif hasattr(model, "decision_function"):
                y_score = model.decision_function(X_test_combined)
                # Convert to probability-like scores for SVM
                y_score = 1 / (1 + np.exp(-y_score))
            else:
                y_score = None
        
        predictions[name] = y_pred
        
        # Calculate all metrics
        accuracy = accuracy_score(y_test, y_pred)
        
        # Per-class metrics
        precision_per_class, recall_per_class, f1_per_class, support_per_class = precision_recall_fscore_support(
            y_test, y_pred, labels=expected_classes, zero_division=0
        )
        
        # Macro averages
        precision_macro = precision_score(y_test, y_pred, average='macro', zero_division=0)
        recall_macro = recall_score(y_test, y_pred, average='macro', zero_division=0)
        f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
        
        # Weighted averages
        precision_weighted = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall_weighted = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        
        # Calculate ROC AUC
        roc_auc = None
        if y_score is not None:
            try:
                y_test_bin = label_binarize(y_test, classes=expected_classes)
                if y_score.shape[1] == len(expected_classes):
                    roc_auc = {}
                    for i, class_name in enumerate(expected_classes):
                        if len(np.unique(y_test_bin[:, i])) > 1:
                            roc_auc[class_name] = roc_auc_score(y_test_bin[:, i], y_score[:, i])
                        else:
                            roc_auc[class_name] = None
                    # Calculate macro average
                    valid_auc = [v for v in roc_auc.values() if v is not None]
                    if valid_auc:
                        roc_auc['macro_avg'] = np.mean(valid_auc)
                    else:
                        roc_auc['macro_avg'] = None
            except Exception as e:
                print(f"  Could not calculate ROC AUC: {e}")
                roc_auc = None
        
        roc_auc_results[name] = roc_auc
        
        # Store results
        results[name] = {
            'accuracy': accuracy,
            'precision_macro': precision_macro,
            'recall_macro': recall_macro,
            'f1_macro': f1_macro,
            'precision_weighted': precision_weighted,
            'recall_weighted': recall_weighted,
            'f1_weighted': f1_weighted,
            'roc_auc_macro': roc_auc['macro_avg'] if roc_auc and 'macro_avg' in roc_auc else None,
            'training_time': training_time,
            'model': model
        }
        
        # Store detailed per-class metrics
        detailed_metrics[name] = {
            'class': expected_classes,
            'precision': precision_per_class,
            'recall': recall_per_class,
            'f1_score': f1_per_class,
            'support': support_per_class
        }
        
        # Print results
        print(f"\n📊 Performance Metrics:")
        print(f"   Accuracy:      {accuracy:.4f}")
        print(f"   Training Time: {training_time:.2f} seconds")
        print(f"\n   Macro Average:")
        print(f"     Precision:   {precision_macro:.4f}")
        print(f"     Recall:      {recall_macro:.4f}")
        print(f"     F1-Score:    {f1_macro:.4f}")
        if roc_auc and 'macro_avg' in roc_auc and roc_auc['macro_avg']:
            print(f"     ROC AUC:     {roc_auc['macro_avg']:.4f}")
        
    except Exception as e:
        print(f"  Error: {e}")
        import traceback
        traceback.print_exc()

# ==================== 12. MODEL COMPARISON TABLE ====================
print("\n" + "=" * 70)
print("STEP 12: MODEL COMPARISON TABLE")
print("=" * 70)

# Create comparison dataframe
comparison_df = pd.DataFrame({
    name: {
        'Accuracy': metrics['accuracy'],
        'Precision_Macro': metrics['precision_macro'],
        'Recall_Macro': metrics['recall_macro'],
        'F1_Macro': metrics['f1_macro'],
        'ROC_AUC': metrics['roc_auc_macro'] if metrics['roc_auc_macro'] else 0,
        'Time (s)': metrics['training_time']
    }
    for name, metrics in results.items()
}).T

# Sort by F1-Macro
comparison_df = comparison_df.sort_values('F1_Macro', ascending=False)

# Display comprehensive table
print("\n" + "=" * 100)
print("COMPREHENSIVE MODEL PERFORMANCE COMPARISON TABLE")
print("=" * 100)
print(comparison_df.round(4).to_string())
print("=" * 100)

# Save table to CSV
comparison_df.to_csv(os.path.join(output_dir, 'table1_model_comparison.csv'))
print(f"✓ Table 1 saved as '{output_dir}/table1_model_comparison.csv'")

# ==================== 13. VISUALIZATION: MODEL COMPARISON ====================
print("\n" + "=" * 70)
print("STEP 13: MODEL COMPARISON VISUALIZATION")
print("=" * 70)

# Figure 3: Model Comparison Bar Chart
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

# Plot 1: Performance metrics comparison
ax1 = axes[0]
x = np.arange(len(comparison_df.index))
width = 0.2

metrics_to_plot = ['Accuracy', 'Precision_Macro', 'Recall_Macro', 'F1_Macro']
colors_metrics = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

for i, (metric, color) in enumerate(zip(metrics_to_plot, colors_metrics)):
    values = comparison_df[metric].values
    bars = ax1.bar(x + i*width, values, width, label=metric.replace('_', ' '), color=color, edgecolor='black', alpha=0.8)
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.2f}', ha='center', va='bottom', fontsize=8, rotation=90)

ax1.set_xlabel('Models', fontsize=12, fontweight='bold')
ax1.set_ylabel('Score', fontsize=12, fontweight='bold')
ax1.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax1.set_xticks(x + width * 1.5)
ax1.set_xticklabels(comparison_df.index, rotation=45, ha='right')
ax1.legend(loc='lower right')
ax1.set_ylim(0, 1.1)
ax1.grid(True, alpha=0.3, axis='y')

# Plot 2: ROC AUC Comparison
ax2 = axes[1]
roc_values = comparison_df['ROC_AUC'].values
valid_roc = [i for i, v in enumerate(roc_values) if v > 0]
valid_roc_values = [roc_values[i] for i in valid_roc]
valid_roc_labels = [comparison_df.index[i] for i in valid_roc]

if valid_roc:
    bars = ax2.bar(range(len(valid_roc)), valid_roc_values, color='#9b59b6', edgecolor='black')
    ax2.set_xlabel('Models', fontsize=12, fontweight='bold')
    ax2.set_ylabel('ROC AUC Score', fontsize=12, fontweight='bold')
    ax2.set_title('ROC AUC Comparison', fontsize=14, fontweight='bold')
    ax2.set_xticks(range(len(valid_roc)))
    ax2.set_xticklabels(valid_roc_labels, rotation=45, ha='right')
    ax2.set_ylim(0, 1.1)
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bar, val in zip(bars, valid_roc_values):
        ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# Plot 3: Training time comparison
ax3 = axes[2]
times = comparison_df['Time (s)'].values
bars = ax3.bar(range(len(times)), times, color='#e67e22', edgecolor='black')
ax3.set_xlabel('Models', fontsize=12, fontweight='bold')
ax3.set_ylabel('Training Time (seconds)', fontsize=12, fontweight='bold')
ax3.set_title('Model Training Time Comparison', fontsize=14, fontweight='bold')
ax3.set_xticks(range(len(times)))
ax3.set_xticklabels(comparison_df.index, rotation=45, ha='right')
ax3.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, time in zip(bars, times):
    ax3.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
             f'{time:.1f}s', ha='center', va='bottom', fontsize=9)

# Plot 4: Individual vs Ensemble comparison
ax4 = axes[3]
individual_models_list = ['Random Forest', 'Decision Tree', 'SVM', 'Naive Bayes']
ensemble_models_list = ['Voting (Hard)', 'Stacking', 'AdaBoost (DT)', 'Gradient Boosting']

individual_scores = [results[m]['f1_macro'] for m in individual_models_list if m in results]
ensemble_scores = [results[m]['f1_macro'] for m in ensemble_models_list if m in results]

if individual_scores and ensemble_scores:
    avg_individual = np.mean(individual_scores)
    avg_ensemble = np.mean(ensemble_scores)
    
    bars = ax4.bar(['Individual Models (Avg)', 'Ensemble Models (Avg)'], 
                   [avg_individual, avg_ensemble],
                   color=['#3498db', '#e74c3c'], edgecolor='black')
    ax4.set_ylabel('F1-Macro Score', fontsize=12, fontweight='bold')
    ax4.set_title('Individual vs Ensemble Performance', fontsize=14, fontweight='bold')
    ax4.set_ylim(0, 1.1)
    ax4.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bar, val in zip(bars, [avg_individual, avg_ensemble]):
        ax4.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'figure3_model_comparison_detailed.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Figure 3 saved as '{output_dir}/figure3_model_comparison_detailed.png'")

# ==================== 14. STEP 3: SELECT BEST PERFORMING MODEL ====================
print("\n" + "=" * 70)
print("STEP 14: SELECTING BEST PERFORMING MODEL")
print("=" * 70)

# Create combined score (40% F1-Macro, 30% ROC AUC, 30% Speed)
comparison_df['Speed_Score'] = 1 / (comparison_df['Time (s)'] + 1)
comparison_df['Combined_Score'] = (
    0.4 * comparison_df['F1_Macro'] + 
    0.3 * comparison_df['ROC_AUC'] + 
    0.3 * comparison_df['Speed_Score']
)

# Find best model by combined score
best_model_name = comparison_df['Combined_Score'].idxmax()
best_model = results[best_model_name]['model']

print(f"\n✅ BEST MODEL SELECTED: {best_model_name}")
print(f"   ====================")
print(f"   F1-Macro:      {results[best_model_name]['f1_macro']:.4f}")
print(f"   Accuracy:      {results[best_model_name]['accuracy']:.4f}")
print(f"   Precision:     {results[best_model_name]['precision_macro']:.4f}")
print(f"   Recall:        {results[best_model_name]['recall_macro']:.4f}")
print(f"   ROC AUC:       {results[best_model_name]['roc_auc_macro']:.4f}")
print(f"   Training Time: {results[best_model_name]['training_time']:.2f} seconds")

# Also find best by pure F1-Macro
best_f1_name = comparison_df['F1_Macro'].idxmax()
print(f"\n🏆 Best by F1-Macro: {best_f1_name}")
print(f"   F1-Macro: {results[best_f1_name]['f1_macro']:.4f}")

# Find best by ROC AUC
best_roc_name = comparison_df['ROC_AUC'].idxmax()
print(f"📈 Best by ROC AUC: {best_roc_name}")
print(f"   ROC AUC: {results[best_roc_name]['roc_auc_macro']:.4f}")

# Find fastest model
fastest_name = comparison_df['Time (s)'].idxmin()
print(f"⚡ Fastest Model: {fastest_name}")
print(f"   Time: {results[fastest_name]['training_time']:.2f}s")

# ==================== 15. CONFUSION MATRIX FOR BEST MODEL ====================
print("\n" + "=" * 70)
print("STEP 15: CONFUSION MATRIX FOR BEST MODEL")
print("=" * 70)

y_pred_best = predictions[best_model_name]
cm = confusion_matrix(y_test, y_pred_best)
labels = expected_classes

# Figure 4: Confusion Matrix Heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Absolute confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels,
            annot_kws={'size': 12}, cbar_kws={'label': 'Count'}, ax=axes[0])
axes[0].set_title(f'Confusion Matrix - {best_model_name} (Absolute)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted Label', fontsize=12)
axes[0].set_ylabel('True Label', fontsize=12)

# Normalized confusion matrix
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='YlOrRd', xticklabels=labels, yticklabels=labels,
            annot_kws={'size': 12}, cbar_kws={'label': 'Proportion'}, ax=axes[1])
axes[1].set_title(f'Confusion Matrix - {best_model_name} (Normalized)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted Label', fontsize=12)
axes[1].set_ylabel('True Label', fontsize=12)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'figure4_confusion_matrix_best_model.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Figure 4 saved as '{output_dir}/figure4_confusion_matrix_best_model.png'")

# ==================== 16. DETAILED CLASSIFICATION REPORT ====================
print("\n" + "=" * 70)
print("STEP 16: DETAILED CLASSIFICATION REPORT")
print("=" * 70)

# Get classification report as dictionary
report = classification_report(y_test, y_pred_best, target_names=labels, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).transpose()

# Display table
print("\n" + "=" * 70)
print("DETAILED CLASSIFICATION REPORT TABLE")
print("=" * 70)
print(report_df.round(4))
print("=" * 70)

# Save to CSV
report_df.to_csv(os.path.join(output_dir, 'table2_classification_report.csv'))
print(f"✓ Table 2 saved as '{output_dir}/table2_classification_report.csv'")

# ==================== 17. SAMPLE PREDICTIONS ====================
print("\n" + "=" * 70)
print("STEP 17: SAMPLE PREDICTIONS")
print("=" * 70)

# Create sample predictions table
test_vader_scores = [sia.polarity_scores(text)['compound'] for text in X_test]
sample_indices = np.random.choice(len(X_test), size=min(20, len(X_test)), replace=False)
sample_df = pd.DataFrame({
    'Review': [X_test.iloc[i][:100] + '...' if len(X_test.iloc[i]) > 100 else X_test.iloc[i] for i in sample_indices],
    'True Label': y_test.iloc[sample_indices].values,
    'Predicted Label': y_pred_best[sample_indices],
    'VADER Score': [test_vader_scores[i] for i in sample_indices]
})

print("\n" + "=" * 100)
print("SAMPLE PREDICTIONS TABLE")
print("=" * 100)
print(sample_df.to_string())
print("=" * 100)

# Save to CSV
sample_df.to_csv(os.path.join(output_dir, 'table3_sample_predictions.csv'), index=False)
print(f"✓ Table 3 saved as '{output_dir}/table3_sample_predictions.csv'")

# ==================== 18. STEP 3: PREDICTION AND DEPLOYMENT ====================
print("\n" + "=" * 70)
print("STEP 18: PREDICTION AND DEPLOYMENT")
print("=" * 70)

# Save best model and components
print(f"\nSaving best model: {best_model_name}")
joblib.dump(best_model, os.path.join(output_dir, 'best_sentiment_model.pkl'))
joblib.dump(tfidf_vectorizer, os.path.join(output_dir, 'tfidf_vectorizer.pkl'))
joblib.dump(feature_extractor, os.path.join(output_dir, 'feature_extractor.pkl'))
joblib.dump(nb_tfidf, os.path.join(output_dir, 'nb_tfidf_vectorizer.pkl'))

print(f"✓ Model and vectorizers saved in '{output_dir}' directory")

# ==================== 18.1 TEST PREDICTION FUNCTION ====================
print("\n" + "-" * 50)
print("18.1 Testing Prediction Function with Sample Reviews")
print("-" * 50)

# Test with sample reviews
test_reviews = [
    "This was an amazing flight! The staff were friendly and the food was delicious.",
    "The flight was okay, nothing special but nothing terrible either.",
    "Terrible experience! The flight was delayed for 5 hours with no explanation.",
    "I felt threatened by the staff's aggressive behavior. They shouted at passengers and made us feel unsafe. This is absolutely unacceptable and violent conduct!"
]

print("\n📝 Testing Prediction Function:")
for i, review in enumerate(test_reviews):
    # Pass nb_tfidf to handle Naive Bayes models
    result = predict_new_review(review, best_model, tfidf_vectorizer, feature_extractor, expected_classes, nb_tfidf)
    print(f"\n{i+1}. Review: {review[:80]}...")
    print(f"   Predicted: {result['predicted_sentiment']}")
    print(f"   VADER Score: {result['vader_score']:.3f}")
    print(f"   Model Type: {result.get('model_type', 'Unknown')}")
    if result['probabilities']:
        probs_str = ", ".join([f"{k}: {v:.2f}" for k, v in result['probabilities'].items()])
        print(f"   Probabilities: {probs_str}")

# ==================== 18.2 DEPLOYMENT FUNCTION ====================
print("\n" + "-" * 50)
print("18.2 Deployment Function for Real-Time Sentiment Analysis")
print("-" * 50)

def deploy_sentiment_analyzer(model_path='sentiment_analysis_results/best_sentiment_model.pkl',
                              tfidf_path='sentiment_analysis_results/tfidf_vectorizer.pkl',
                              feature_path='sentiment_analysis_results/feature_extractor.pkl',
                              nb_tfidf_path='sentiment_analysis_results/nb_tfidf_vectorizer.pkl'):
    """
    Deploy the sentiment analyzer for real-time predictions
    """
    print("🔄 Loading deployed model...")
    
    # Load model and components
    model = joblib.load(model_path)
    tfidf = joblib.load(tfidf_path)
    feature_ext = joblib.load(feature_path)
    nb_tfidf = joblib.load(nb_tfidf_path)
    
    print("✅ Model loaded successfully!")
    print("📊 Ready for real-time sentiment analysis")
    
    def analyze(review_text):
        """
        Analyze a single review in real-time
        """
        result = predict_new_review(review_text, model, tfidf, feature_ext, expected_classes, nb_tfidf)
        return result
    
    def analyze_batch(reviews_list):
        """
        Analyze a batch of reviews
        """
        results = []
        for review in reviews_list:
            results.append(analyze(review))
        return results
    
    return analyze, analyze_batch

# Create deployment function
print("\n📦 Creating deployment functions...")
analyze_single, analyze_batch = deploy_sentiment_analyzer()

# Test deployment with a new review
print("\n🎯 Testing Deployment with New Review:")
new_review = "The cabin crew was extremely rude and made threatening comments. I've never felt so unsafe on a flight!"
result = analyze_single(new_review)
print(f"Review: {new_review[:80]}...")
print(f"Prediction: {result['predicted_sentiment']}")
print(f"VADER Score: {result['vader_score']:.3f}")
print(f"Model Type: {result.get('model_type', 'Unknown')}")

# ==================== 19. SUMMARY DASHBOARD ====================
print("\n" + "=" * 70)
print("STEP 19: CREATING SUMMARY DASHBOARD")
print("=" * 70)

# Create comprehensive summary dashboard
fig = plt.figure(figsize=(20, 16))

# 1. Class distribution comparison
ax1 = plt.subplot(3, 4, 1)
before_counts = label_counts
after_counts = balanced_counts
x = range(4)
width = 0.35
ax1.bar([i - width/2 for i in x], before_counts.values, width, label='Before', color='lightgray', edgecolor='black')
ax1.bar([i + width/2 for i in x], after_counts.values, width, label='After', color=colors, edgecolor='black')
ax1.set_title('Class Distribution: Before vs After', fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(before_counts.index, rotation=45)
ax1.set_ylabel('Count')
ax1.legend()

# 2. Best model metrics
ax2 = plt.subplot(3, 4, 2)
best_metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Macro', 'ROC AUC']
best_values = [
    results[best_model_name]['accuracy'],
    results[best_model_name]['precision_macro'],
    results[best_model_name]['recall_macro'],
    results[best_model_name]['f1_macro'],
    results[best_model_name]['roc_auc_macro'] if results[best_model_name]['roc_auc_macro'] else 0
]
bars = ax2.bar(best_metrics, best_values, color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6'], edgecolor='black')
ax2.set_title(f'Best Model: {best_model_name[:20]}', fontweight='bold')
ax2.set_ylim(0, 1)
ax2.set_ylabel('Score')
ax2.tick_params(axis='x', rotation=45)
for bar, val in zip(bars, best_values):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# 3. Confusion matrix (normalized)
ax3 = plt.subplot(3, 4, 3)
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='YlOrRd', 
            xticklabels=labels, yticklabels=labels, ax=ax3, cbar=False)
ax3.set_title('Normalized Confusion Matrix', fontweight='bold')

# 4. Per-class performance for best model
ax4 = plt.subplot(3, 4, 4)
per_class_data = detailed_metrics[best_model_name]
x = range(len(labels))
width = 0.25
ax4.bar([i - width for i in x], per_class_data['precision'], width, label='Precision', color='#3498db')
ax4.bar(x, per_class_data['recall'], width, label='Recall', color='#2ecc71')
ax4.bar([i + width for i in x], per_class_data['f1_score'], width, label='F1', color='#e74c3c')
ax4.set_title(f'Per-Class Metrics - {best_model_name}', fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels(labels, rotation=45)
ax4.set_ylim(0, 1)
ax4.legend(fontsize=8)

# 5. Individual vs Ensemble comparison
ax5 = plt.subplot(3, 4, 5)
individual_scores = [results[m]['f1_macro'] for m in individual_models_list if m in results]
ensemble_scores = [results[m]['f1_macro'] for m in ensemble_models_list if m in results]
if individual_scores and ensemble_scores:
    ax5.bar(['Individual', 'Ensemble'], [np.mean(individual_scores), np.mean(ensemble_scores)],
            color=['#3498db', '#e74c3c'], edgecolor='black')
    ax5.set_title('Individual vs Ensemble (Avg F1)', fontweight='bold')
    ax5.set_ylim(0, 1)
    ax5.set_ylabel('F1-Macro')

# 6. ROC AUC comparison
ax6 = plt.subplot(3, 4, 6)
valid_models = [name for name in comparison_df.index if results[name]['roc_auc_macro']]
valid_auc = [results[name]['roc_auc_macro'] for name in valid_models]
if valid_models:
    top_auc_idx = np.argsort(valid_auc)[-5:]  # Top 5
    top_auc_models = [valid_models[i] for i in top_auc_idx]
    top_auc_values = [valid_auc[i] for i in top_auc_idx]
    bars = ax6.barh(range(len(top_auc_models)), top_auc_values, color='#9b59b6', edgecolor='black')
    ax6.set_yticks(range(len(top_auc_models)))
    ax6.set_yticklabels([m[:15] for m in top_auc_models])
    ax6.set_title('Top 5 Models by ROC AUC', fontweight='bold')
    ax6.set_xlim(0, 1)
    for bar, val in zip(bars, top_auc_values):
        ax6.text(bar.get_width() - 0.05, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', ha='right', va='center', fontsize=9)

# 7. Training time comparison
ax7 = plt.subplot(3, 4, 7)
fastest_models = comparison_df.nsmallest(5, 'Time (s)')
bars = ax7.barh(range(len(fastest_models)), fastest_models['Time (s)'].values, color='#e67e22', edgecolor='black')
ax7.set_yticks(range(len(fastest_models)))
ax7.set_yticklabels([m[:15] for m in fastest_models.index])
ax7.set_title('Fastest Models (Training Time)', fontweight='bold')
ax7.set_xlabel('Seconds')
for bar, time in zip(bars, fastest_models['Time (s)'].values):
    ax7.text(bar.get_width() - 0.5, bar.get_y() + bar.get_height()/2,
             f'{time:.1f}s', ha='right', va='center', fontsize=9)

# 8. Violent class analysis
ax8 = plt.subplot(3, 4, 8)
violent_idx = labels.index('violent')
violent_metrics = [
    detailed_metrics[best_model_name]['precision'][violent_idx],
    detailed_metrics[best_model_name]['recall'][violent_idx],
    detailed_metrics[best_model_name]['f1_score'][violent_idx]
]
bars = ax8.bar(['Precision', 'Recall', 'F1'], violent_metrics, color='#8e44ad', edgecolor='black')
ax8.set_title('Violent Class Performance', fontweight='bold')
ax8.set_ylim(0, 1)
for bar, val in zip(bars, violent_metrics):
    ax8.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', va='bottom', fontsize=10)

# 9. Deployment info
ax9 = plt.subplot(3, 4, 9)
deployment_text = f"""
📦 DEPLOYMENT READY

Best Model: {best_model_name}
Saved to: best_sentiment_model.pkl

Prediction Function:
analyze_single(review_text)
analyze_batch(reviews_list)

Sample Usage:
>>> result = analyze_single(
    "Great flight!"
)
>>> print(result['predicted_sentiment'])
"""
ax9.text(0.05, 0.5, deployment_text, transform=ax9.transAxes, fontsize=10,
         verticalalignment='center', fontfamily='monospace')
ax9.set_title('Deployment Information', fontweight='bold')
ax9.axis('off')

# 10. Dataset statistics
ax10 = plt.subplot(3, 4, 10)
stats_text = f"""
📊 DATASET STATISTICS

Original samples: {len(df)}
After cleaning: {len(df_clean)}
After balancing: {len(df_balanced)}
Training set: {len(X_train)}
Test set: {len(X_test)}

Class distribution (balanced):
Positive: {balanced_counts.get('positive', 0)}
Neutral:  {balanced_counts.get('neutral', 0)}
Negative: {balanced_counts.get('negative', 0)}
Violent:  {balanced_counts.get('violent', 0)}
"""
ax10.text(0.05, 0.5, stats_text, transform=ax10.transAxes, fontsize=10,
          verticalalignment='center', fontfamily='monospace')
ax10.set_title('Dataset Statistics', fontweight='bold')
ax10.axis('off')

# 11. Model ranking
ax11 = plt.subplot(3, 4, 11)
ranking_text = "🏆 MODEL RANKING (by F1-Macro)\n" + "-"*25 + "\n"
for i, (name, row) in enumerate(comparison_df.head(5).iterrows()):
    ranking_text += f"{i+1}. {name[:20]}: {row['F1_Macro']:.4f}\n"
ax11.text(0.05, 0.5, ranking_text, transform=ax11.transAxes, fontsize=10,
          verticalalignment='center', fontfamily='monospace')
ax11.set_title('Top 5 Models', fontweight='bold')
ax11.axis('off')

# 12. Configuration
ax12 = plt.subplot(3, 4, 12)
config_text = f"""
⚙️ CONFIGURATION

Violent Threshold: {VIOLENT_THRESHOLD}
Positive Threshold: > {POSITIVE_THRESHOLD}
Negative Threshold: < {NEGATIVE_THRESHOLD}

Features:
- TF-IDF (2000 features)
- Text statistics (7 features)

Balancing: Manual Resampling
Target count: {len(df_balanced)//4}
"""
ax12.text(0.05, 0.5, config_text, transform=ax12.transAxes, fontsize=10,
          verticalalignment='center', fontfamily='monospace')
ax12.set_title('Configuration', fontweight='bold')
ax12.axis('off')

plt.suptitle('SENTIMENT ANALYSIS WITH VIOLENCE DETECTION - COMPLETE DASHBOARD', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'figure5_complete_dashboard.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Figure 5 saved as '{output_dir}/figure5_complete_dashboard.png'")

# ==================== 20. FINAL SUMMARY ====================
print("\n" + "=" * 70)
print("PROCESS COMPLETED SUCCESSFULLY")
print("=" * 70)
print("\n📊 FINAL SUMMARY")
print("-" * 50)
print(f"\n✅ Classes: {expected_classes}")
print(f"\n✅ BEST MODEL: {best_model_name}")
print(f"   - Accuracy:      {results[best_model_name]['accuracy']:.4f}")
print(f"   - Precision:     {results[best_model_name]['precision_macro']:.4f}")
print(f"   - Recall:        {results[best_model_name]['recall_macro']:.4f}")
print(f"   - F1-Macro:      {results[best_model_name]['f1_macro']:.4f}")
print(f"   - ROC AUC:       {results[best_model_name]['roc_auc_macro']:.4f}")
print(f"   - Training Time: {results[best_model_name]['training_time']:.2f}s")

print("\n📁 FILES GENERATED:")
print(f"   - newbalanceddata.csv")
print(f"   - {output_dir}/table1_model_comparison.csv")
print(f"   - {output_dir}/table2_classification_report.csv")
print(f"   - {output_dir}/table3_sample_predictions.csv")
print(f"   - {output_dir}/figure1_class_distribution_before.png")
print(f"   - {output_dir}/figure2_class_distribution_after.png")
print(f"   - {output_dir}/figure3_model_comparison_detailed.png")
print(f"   - {output_dir}/figure4_confusion_matrix_best_model.png")
print(f"   - {output_dir}/figure5_complete_dashboard.png")
print(f"   - {output_dir}/best_sentiment_model.pkl")
print(f"   - {output_dir}/tfidf_vectorizer.pkl")
print(f"   - {output_dir}/feature_extractor.pkl")
print(f"   - {output_dir}/nb_tfidf_vectorizer.pkl")

print("\n🚀 DEPLOYMENT READY:")
print("   Use the following functions for real-time predictions:")
print("   >>> from sentiment_analyzer import analyze_single, analyze_batch")
print("   >>> result = analyze_single('Your review text here')")
print("   >>> print(result['predicted_sentiment'])")

print("\n" + "=" * 70)

Scikit-learn version: 1.0.2
Number of CPU cores: 4


STEP 1: LOADING AND EXPLORING DATA
Original shape: (3629, 22)

Columns in dataset:
  1. 'airline'
  2. 'cabin'
  3. 'cabin_service'
  4. 'date_flown_day'
  5. 'date_flown_month'
  6. 'date_flown_year'
  7. 'entertainment'
  8. 'food_bev'
  9. 'ground_service'
  10. 'has_layover'
  11. 'pos_neu_neg_review_score'
  12. 'recommended'
  13. 'review_characters'
  14. 'review_date_date_flown_distance_days'
  15. 'review_date_day'
  16. 'review_date_month'
  17. 'review_date_year'
  18. 'review_score'
  19. 'review_text'
  20. 'seat_comfort'
  21. 'traveller_type'
  22. 'value_for_money'

STEP 2: IMPLEMENTING ALGORITHM 1 - VADER SENTIMENT WITH VIOLENCE THRESHOLD
Classes: ['positive', 'neutral', 'negative', 'violent']
Violent = Very Negative (VADER compound score < -0.5)

Analyzing VADER score distribution...
VADER score statistics:
  Mean: 0.259
  Std: 0.776
  Min: -0.996
  Max: 0.999
  25th percentile: -0.643
  75th percentile: 0.954

Labe

<Figure size 1400x600 with 2 Axes>

✓ Figure 1 saved as 'sentiment_analysis_results/figure1_class_distribution_before.png'

STEP 4: VERIFYING CLASSES
Expected classes: ['positive', 'neutral', 'negative', 'violent']
Actual classes: ['negative', 'neutral', 'positive', 'violent']
✅ Class verification passed: Exactly 4 classes

STEP 5: PREPARING DATA FOR BALANCING
After dropping NA: (3628, 2)

Preprocessing text...
After removing very short reviews: (3628, 3)
After removing duplicates: (3628, 3)

Class distribution before balancing:
positive    2265
violent     1023
negative     282
neutral       58
Name: label, dtype: int64

Imbalance ratio (max/min): 39.05

STEP 6: BALANCING THE DATA

Using manual resampling for class balancing...
Original class distribution: {'positive': 2265, 'violent': 1023, 'negative': 282, 'neutral': 58}
Target count per class: 2265
Balanced class distribution: {'positive': 2265, 'negative': 2265, 'violent': 2265, 'neutral': 2265}

Balanced dataset shape: (9060, 3)
Balanced class distribution:
positiv

<Figure size 1400x600 with 2 Axes>

✓ Figure 2 saved as 'sentiment_analysis_results/figure2_class_distribution_after.png'

STEP 8: TRAIN-TEST SPLIT (USING BALANCED DATA)
Unique classes: ['negative', 'neutral', 'positive', 'violent']
Number of classes: 4
Training set size: 7248
Test set size: 1812

Training set class distribution:
positive    1812
negative    1812
violent     1812
neutral     1812
Name: label, dtype: int64

Test set class distribution:
violent     453
positive    453
neutral     453
negative    453
Name: label, dtype: int64

STEP 9: FEATURE EXTRACTION
Extracting: TF-IDF + basic extra features (length, VADER scores)
Transforming training data...
Training features shape: (7248, 2007)
Time taken: 15.24 seconds

Transforming test data...
Test features shape: (1812, 2007)
Time taken: 3.69 seconds

STEP 10: MODEL TRAINING AND ENSEMBLE LEARNING
Training individual classifiers: RF, DT, SVM, NB
Training ensemble models: Voting, Stacking, Boosting

--------------------------------------------------
10.1 Individual 

...

✓ Figure 3 saved as 'sentiment_analysis_results/figure3_model_comparison_detailed.png'

STEP 14: SELECTING BEST PERFORMING MODEL

✅ BEST MODEL SELECTED: Naive Bayes
   F1-Macro:      0.8172
   Accuracy:      0.8179
   Precision:     0.8222
   Recall:        0.8179
   ROC AUC:       0.7123
   Training Time: 0.03 seconds

🏆 Best by F1-Macro: Stacking
   F1-Macro: 0.9856
📈 Best by ROC AUC: Random Forest
   ROC AUC: 0.7145
⚡ Fastest Model: Naive Bayes
   Time: 0.03s

STEP 15: CONFUSION MATRIX FOR BEST MODEL


<Figure size 1600x700 with 4 Axes>

✓ Figure 4 saved as 'sentiment_analysis_results/figure4_confusion_matrix_best_model.png'

STEP 16: DETAILED CLASSIFICATION REPORT

DETAILED CLASSIFICATION REPORT TABLE
              f1-score  precision  recall    support
positive        0.7415     0.8283  0.6711   453.0000
neutral         0.9978     0.9956  1.0000   453.0000
negative        0.7814     0.7490  0.8168   453.0000
violent         0.7482     0.7157  0.7837   453.0000
accuracy        0.8179     0.8179  0.8179     0.8179
macro avg       0.8172     0.8222  0.8179  1812.0000
weighted avg    0.8172     0.8222  0.8179  1812.0000
✓ Table 2 saved as 'sentiment_analysis_results/table2_classification_report.csv'

STEP 17: SAMPLE PREDICTIONS

SAMPLE PREDICTIONS TABLE
                                               Review True Label Predicted Label  VADER Score
0   however recent istkul flight last check counte...    violent        negative      -0.8271
1   lounge simply best especially krug champagne p...   positive        positive    

<Figure size 2000x1600 with 12 Axes>

✓ Figure 5 saved as 'sentiment_analysis_results/figure5_complete_dashboard.png'

PROCESS COMPLETED SUCCESSFULLY

📊 FINAL SUMMARY
--------------------------------------------------

✅ Classes: ['positive', 'neutral', 'negative', 'violent']

✅ BEST MODEL: Naive Bayes
   - Accuracy:      0.8179
   - Precision:     0.8222
   - Recall:        0.8179
   - F1-Macro:      0.8172
   - ROC AUC:       0.7123
   - Training Time: 0.03s

📁 FILES GENERATED:
   - newbalanceddata.csv
   - sentiment_analysis_results/table1_model_comparison.csv
   - sentiment_analysis_results/table2_classification_report.csv
   - sentiment_analysis_results/table3_sample_predictions.csv
   - sentiment_analysis_results/figure1_class_distribution_before.png
   - sentiment_analysis_results/figure2_class_distribution_after.png
   - sentiment_analysis_results/figure3_model_comparison_detailed.png
   - sentiment_analysis_results/figure4_confusion_matrix_best_model.png
   - sentiment_analysis_results/figure5_complete_dashboard.p